In [4]:
from Z import *
from Poly import *
import numpy as np

In [5]:
''' Some number theoretic helper functions'''
def factor(n : int) -> list[list[int]]:
    assert n >= 1, 'factoring is defined for positive integers'
    result = []
    num = n
    try:
        with open('small_primes.txt') as file:
            for prime in file:
                p = int(prime[:-1])
                if num % p == 0:
                    div = [p, 1]
                    num = num // p
                    while num % p == 0:
                        num //= p
                        div[1] += 1
                    result.append(div)
                if num == 1: 
                    return result
            raise ValueError('too big to factor by trial division')
    except:
        if is_prime(num):
            return [[num, 1]]
        elif p_n := prime_power(num):
            assert type(p_n) == list
            return [p_n]
        else:
            # smallest number not factorable this way is > 4 * 10^8 
            raise ValueError('too big to factor by trial division')
    
def mobius(n : int) -> int:
    assert n >= 1, 'mobius function is defined for positive integers'
    factors = factor(n)
    if any([exp > 1 for prime, exp in factors]):
        return 0
    else:
        return (-1) ** (len(factors) % 2)

def phi(n : int) -> int:
    assert n >= 1, 'totient function is defined for positive integers'
    factors = factor(n)
    result = 1
    for prime, exp in factors:
        result *= (prime - 1) * prime ** (exp - 1)
    return result

def rad(n : int) -> int:
    assert n >= 1, 'radical function is defined for positive integers'
    factors = factor(n)
    result = 1
    for prime, exp in factors:
        result *= prime
    return result

def divs(n : int) -> list[int]:
    assert n >= 1, 'divisors are defined for positive integers'
    if n == 1: return [1]
    factors = factor(n)
    p, e = factors[0]
    divisors = [p ** k for k in range(e + 1)]
    for prime, exp in factors[1:]:
        more_divs = [prime ** k * div for div in divisors 
                     for k in range(1, exp + 1)]
        divisors += more_divs
    return sorted(divisors)

def val(n : int, p : int) -> int:
    ''' max {k | p^k divides n}'''
    assert n >= 1, 'valuation is defined for positive integers'
    v, m = 0, n
    while m % p == 0:
        m //= p
        v += 1
    return v

def gcd(n : int, m : int) -> list[int]:
    ''' Find the bezout coefficents x * n + y * m = gcd
    returns [x, y, gcd]'''
    # Initialize: [x, y, value]
    x0, y0, r0 = 1, 0, n
    x1, y1, r1 = 0, 1, m
    
    while r1 > 0: # do (extended) euclidean algorithm
        q = r0 // r1
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
        r0, r1 = r1, r0 - q * r1        
    return [x0, y0, r0]

def ramanujan_sum(q : int, n : int) -> int:
    ''' returns sum_{gcd(a, q) = 1} e^(2 pi i a n / q), i.e. the sum of 
    the n^th powers of the primitive q^th roots of unity'''
    assert q >= 1, 'nth roots of unity is defined for n >= 1'
    result = 0
    for d in divs(gcd(q, n)[-1]):
        result += d * mobius(q // d)
    return result    

def factorial(n : int) -> int:
    assert n >= 0, 'factorial is defined for non-negative integers'
    prod = 1
    for k in range(2, n + 1):
        prod *= k
    return prod

def binom(n : int, k : int) -> int:
    ''' binomial coefficient n choose k'''
    assert n >= 0, 'n choose k needs n >= 0'
    if k > n or k < 0: 
        return 0
    numerator = 1
    for i in range(k + 1, n + 1):
        numerator *= i
    return numerator // factorial(n - k)

In [6]:
''' Some specific polynomial operations as arrays'''
def compose(poly : np.ndarray, exp : int, neg : bool = False) -> np.ndarray:
    ''' poly(x) -> poly(x ^ exp)'''
    d = len(poly) - 1
    result = np.zeros(exp * d + 1, dtype=poly.dtype) 
    result[::exp] = poly
    if neg:
        result[exp::2 * exp] *= -1
    return result

def unity_mult_mod(poly: np.ndarray, n : int, d : int) -> np.ndarray:
    ''' poly(x) -> poly(x)(1 - x^n) modulo x^d'''
    # make the right length first
    result = poly[:d] if len(poly) >= d else np.pad(poly, (0, d - len(poly)))
    for i in range(d - 1, n - 1, -1):
        result[i] -= result[i - n]
    return result

def unity_div_mod(poly : np.ndarray, n : int, d : int) -> np.ndarray:
    ''' poly(x) -> poly(x)/(1 - x^n) modulo x^d'''
    # make the right length first
    result = poly[:d] if len(poly) >= d else np.pad(poly, (0, d - len(poly)))
    for i in range(n, d):
        result[i] += result[i - n]
    return result

def mult_mod(D : int, *args) -> np.ndarray:
    prod = np.array([int(k == 0) for k in range(D)], dtype=object)
    holder = np.zeros(D, dtype=object)
    for p in args:
        for i in range(D):
            for j in range(i + 1):
                if len(p) > i - j:
                    holder[i] += prod[j] * p[i - j]
        prod = holder
        holder = np.zeros(D, dtype=object)
    return prod
    
def bell(*args) -> int | float | complex:
    ''' evalates the complete bell polynomial B_n(x_1, ... , x_n)'''
    n = len(args)
    B = {0 : 1}
    for k in range(1, n + 1):
        result = 0
        for i in range(k):
            result += binom(k - 1, i) * B[k - i - 1] * args[i]
        B[k] = result
    return B[n]


In [7]:
def cyclo_odd_sq_free_composite(n : int, check=True) -> np.ndarray:
    ''' Phi_n(x) given that n = rad(n) is odd and composite'''
    if check:
        # check that n is of correct form
        factors = factor(n)
        assert (len(factors) > 1 
                and factors[0][0] > 2 
                and all([exp == 1 for p, exp in factors])
                ), 'odd sq free composite means product of distinct odd primes'
    
    D = phi(n) // 2 # is necessarily odd
    result = np.zeros(D + 1, dtype=object) # first half of coeff since symmetric
    result[0] = 1
    for div in divs(n)[:-1]:
        if mobius(n // div) == 1:
            unity_mult_mod(result, div, D + 1) 
        else:
            unity_div_mod(result, div, D + 1)
            
    reverse = result[-2::-1] # first phi(n)/2 - 1 coeff in reverse order
    combined = np.concatenate((result, reverse), axis=0) # palindromic
    return combined

def cyclo(n : int) -> np.ndarray:
    if n == 1: # trivial case
        return np.array([-1, 1], dtype=object)
    N = rad(n)
    if N == 2: # n = 2^k ==> phi_n(x) = x^{2^{k-1}} + 1
        result = [int(k in {0, n//2}) for k in range(1 + n // 2)] 
        return np.array(result, dtype=object)
    elif N % 2 == 0: # n = 2^k * (odd) ==> phi_n(x)=phi_{odd}(-x^{2^{k-1}})
        v = val(n, 2)
        return compose(cyclo(n // 2**v), 2**(v - 1), neg=True)
    elif is_prime(N):
        phi_p = np.array([1 for _ in range(N)], dtype=object)
        v = val(n, N)
        return compose(phi_p, N ** (v - 1))
    else: # phi_n(x) = phi_{rad(n)}(x^{n/rad(n)})
        return compose(cyclo_odd_sq_free_composite(N), n // N)

def A(n : int) -> int:
    return np.max(np.abs(cyclo(n)))

def cyclo_coeff(n : int, k : int) -> int:
    ''' return the coefficient on x^k in Phi(x) '''
    assert n > 0, 'cyclotomic polynomial is defined for n >= 1'
    if n == 1:
        if k == 0: return -1
        elif k == 1: return 1
        else: return 0
    if k in {phi(n), 0}:
        return 1
    if k < 0 or k > phi(n):
        return 0
    if is_prime(n):
        return 1        
    K = min(k, phi(n) - k) # palindromic
    args = [-1 * factorial(j - 1) * ramanujan_sum(n, j) for j in range(1, K+1)]
    return bell(*args) // factorial(K)

In [16]:
n = 105
while A(n) <= 2:
    n += 2
factor(n)
A(n), cyclo(n), n, factor(n)

(3,
 array([1, 1, 1, 1, 1, 0, 0, -1, -1, -1, -1, -2, -1, -1, -1, -1, 0, 0, 1,
        1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0,
        0, -1, -1, -1, -1, -2, -1, -1, -1, -1, 0, 0, 1, 1, 2, 2, 2, 1, 1,
        0, 0, -1, -1, -1, -1, -2, -1, -1, -1, 0, 1, 1, 2, 2, 1, 1, 1, 0, 0,
        0, -1, -1, -1, -2, -2, -1, -1, -1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0,
        -1, -2, -1, -1, -1, 0, 1, 1, 2, 2, 2, 2, 2, 1, 1, 0, -1, -2, -2,
        -3, -3, -3, -2, -2, -1, 0, 1, 1, 2, 2, 2, 2, 2, 1, 1, 0, -1, -1,
        -1, -2, -1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, -1, -1, -1, -2, -2,
        -1, -1, -1, 0, 0, 0, 1, 1, 1, 2, 2, 1, 1, 0, -1, -1, -1, -2, -1,
        -1, -1, -1, 0, 0, 1, 1, 2, 2, 2, 1, 1, 0, 0, -1, -1, -1, -1, -2,
        -1, -1, -1, -1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 1, 1, 1, 1, 0, 0, -1, -1, -1, -1, -2, -1, -1, -1, -1, 0,
        0, 1, 1, 1, 1, 1], dtype=object),
 385,
 [[5, 1], [7, 1], [11, 1]])

In [20]:
for n in range(1, 201):
    odd_factors = factor( n // (1 << val(n, 2)))
    if len(odd_factors) > 2:
        print(n, factor(n))

105 [[3, 1], [5, 1], [7, 1]]
165 [[3, 1], [5, 1], [11, 1]]
195 [[3, 1], [5, 1], [13, 1]]


In [25]:
[(3**k)%15 for k in range(1, 10)]

[3, 9, 12, 6, 3, 9, 12, 6, 3]

In [9]:
def phi_zero(n, D):
    return mult_mod(D, *[cyclo(k) for k in range(1, n+1)])

def x_coeff(n, d):
    prod = np.array([int(k==0) for k in range(d+1)])
    coeff = [prod[d]]
    for k in range(1, n+1):
        prod = mult_mod(d+1, prod, cyclo(k))
        coeff.append(prod[d])
    return coeff

# import matplotlib.pyplot as plt

# plt.plot(x_coeff(20000, 1))
# plt.show()
        

In [26]:
[(n, 2 + int_log(n - 1, 2)) for n in range(2, 10)]

[(2, 2), (3, 3), (4, 3), (5, 4), (6, 4), (7, 4), (8, 4), (9, 5)]